##L08: Data quality rules and a quarantine table

###Goal
Split conformed trips into valid rows (da.silver.yellow_trips) and rejected rows with reasons (da.silver.yellow_trips_quarantine).

###Step 1. Define the rules once
Each rule is a SQL expression that must be true. Update the date window if you load other months.

In [0]:
from pyspark.sql import functions as F

rules = {
  "pickup_before_dropoff": "pickup_ts < dropoff_ts",
  "pickup_in_window":      "pickup_ts >= '2024-12-01' AND pickup_ts < '2025-03-01'",
  "distance_sane":         "trip_distance >= 0 AND trip_distance < 200",
  "fare_non_negative":     "fare_amount >= 0",
  "total_non_negative":    "total_amount >= 0",
  "zones_valid":           "pu_location_id BETWEEN 1 AND 265 AND do_location_id BETWEEN 1 AND 265",
  "passengers_sane":       "passenger_count IS NULL OR passenger_count BETWEEN 0 AND 8",
  "duration_sane":         "trip_minutes BETWEEN 0 AND 1440",
}

###Step 2. Tag every row with the rules it fails
A null result counts as a failure, so wrap each rule in coalesce.

In [0]:
df = spark.table("da.silver.yellow_trips_conformed")
checks = [F.when(~F.coalesce(F.expr(e), F.lit(False)), F.lit(name)) for name, e in rules.items()]
df = df.withColumn("failed_rules", F.filter(F.array(*checks), lambda x: x.isNotNull()))

###Step 3. Split and write both tables

In [0]:
valid = df.filter(F.size("failed_rules") == 0).drop("failed_rules")
bad = df.filter(F.size("failed_rules") > 0)
valid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("da.silver.yellow_trips")
bad.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("da.silver.yellow_trips_quarantine")

###Step 4. Report the damage
Counts per rule (a row can fail more than one rule) and the overall rejection rate.

In [0]:
%sql
SELECT rule, COUNT(*) AS rows_failed
FROM (SELECT explode(failed_rules) AS rule FROM da.silver.yellow_trips_quarantine)
GROUP BY rule ORDER BY rows_failed DESC;
SELECT (SELECT COUNT(*) FROM da.silver.yellow_trips) AS valid,
       (SELECT COUNT(*) FROM da.silver.yellow_trips_quarantine) AS rejected;

   
###Step 5. Document the rules
One row per rule: name, expression, why it exists, and rows failed. Rules with zero failures still appear (LEFT JOIN against the quarantine counts).

In [0]:
%sql
WITH rule_docs(name, expression, why) AS (
  VALUES
    ('pickup_before_dropoff', 'pickup_ts < dropoff_ts', 'A trip cannot end before it starts; reversed timestamps produce negative trip_minutes and poison duration and speed analytics.'),
    ('pickup_in_window', 'pickup_ts >= ''2024-12-01'' AND pickup_ts < ''2025-03-01''', 'Guards the declared load window (Dec 2024 - Feb 2025). TLC files contain stray boundary rows with impossible dates; anything outside the window is source junk.'),
    ('distance_sane', 'trip_distance >= 0 AND trip_distance < 200', 'The NYC taxi zone map spans roughly 30 miles end to end; 200 is generous headroom. Negative distances are meter errors.'),
    ('fare_non_negative', 'fare_amount >= 0', 'A fare is a charge. Negative fares are refunds or voided meters and would drag averages negative.'),
    ('total_non_negative', 'total_amount >= 0', 'Total is what the rider actually pays; negative totals are disputed or cancelled rides, not revenue.'),
    ('zones_valid', 'pu_location_id BETWEEN 1 AND 265 AND do_location_id BETWEEN 1 AND 265', 'The TLC zone lookup only defines LocationIDs 1-265. Anything outside silently drops rows in joins with taxi_zones (L06/L07).'),
    ('passengers_sane', 'passenger_count IS NULL OR passenger_count BETWEEN 0 AND 8', 'Yellow cabs legally seat about 6 (8 = headroom). NULL is allowed because some vendor files omit passenger_count entirely.'),
    ('duration_sane', 'trip_minutes BETWEEN 0 AND 1440', 'Trips longer than 24 hours indicate a stuck meter; 1440 minutes is the longest plausible shift.')
),
failures AS (
  SELECT rule, COUNT(*) AS rows_failed
  FROM (SELECT explode(failed_rules) AS rule FROM da.silver.yellow_trips_quarantine)
  GROUP BY rule
)
SELECT d.name, d.expression, d.why, COALESCE(f.rows_failed, 0) AS rows_failed
FROM rule_docs d
LEFT JOIN failures f ON d.name = f.rule
ORDER BY rows_failed DESC, d.name